# Employee Retention Analysis
**Step 5:** Analyze data in DataFrame using the document

| RQ | Question | Method |
|----|----------|--------|
| RQ1 | Does salary affect retention? | T-Test |
| RQ2 | Does job satisfaction affect retention? | T-Test / Chi-Square |
| RQ3 | Do some departments experience higher turnover? | ANOVA |
| RQ4 | Does workload affect resignation rates? | Chi-Square + Quartiles |
| RQ5 | Which employee groups are highest risk? | Employee Segmentation |

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# stroke.db lives one level up from src/
DB_PATH = os.path.join(os.path.dirname(os.path.abspath('data-analysis.ipynb')), '..', 'stroke.db')
conn = sqlite3.connect(DB_PATH)
print('Connected to', os.path.abspath(DB_PATH))

## Load Data from SQLite into DataFrame

In [ ]:
query = """
SELECT
    p.patient_id,
    p.gender,
    p.age,
    p.ever_married,
    wt.work_type_name      AS work_type,
    rt.residence_type_name AS residence_type,
    ph.hypertension,
    ph.heart_disease,
    ph.avg_glucose_level,
    ph.bmi,
    ss.smoking_status_name AS smoking_status,
    ph.stroke
FROM patient p
JOIN patient_health       ph ON p.patient_id       = ph.patient_id
JOIN dim_work_type        wt ON p.work_type_id      = wt.work_type_id
JOIN dim_residence_type   rt ON p.residence_type_id = rt.residence_type_id
LEFT JOIN dim_smoking_status ss ON ph.smoking_status_id = ss.smoking_status_id
"""
df = pd.read_sql_query(query, conn)
print('Shape:', df.shape)
df.head()

## RQ1: Does Salary (Glucose Level) Affect Retention (Stroke)? — T-Test
> Glucose level is used as the continuous health metric, analogous to salary as a risk driver.

In [ ]:
grp_stroke    = df[df['stroke'] == 1]['avg_glucose_level']
grp_no_stroke = df[df['stroke'] == 0]['avg_glucose_level']

t_stat, p_val = stats.ttest_ind(grp_stroke, grp_no_stroke)

print('RQ1 -- T-Test: Glucose Level vs Stroke Outcome')
print(f'  Stroke group mean    : {grp_stroke.mean():.2f}')
print(f'  No-stroke group mean : {grp_no_stroke.mean():.2f}')
print(f'  T-statistic : {t_stat:.4f}')
print(f'  P-value     : {p_val:.4e}')
print(f'  Result      : {"SIGNIFICANT" if p_val < 0.05 else "NOT significant"} (a=0.05)')

fig, ax = plt.subplots()
ax.boxplot([grp_no_stroke, grp_stroke], tick_labels=['No Stroke', 'Stroke'])
ax.set_title('RQ1 -- Glucose Level by Stroke Outcome (T-Test)')
ax.set_ylabel('Avg Glucose Level')
plt.tight_layout()
plt.show()

## RQ2: Does Job Satisfaction (Hypertension) Affect Retention (Stroke)? — Chi-Square

In [ ]:
ct = pd.crosstab(df['hypertension'], df['stroke'])
chi2, p, dof, _ = stats.chi2_contingency(ct)

print('RQ2 -- Chi-Square: Hypertension vs Stroke')
print(ct)
print(f'  Chi2={chi2:.4f}  p={p:.4e}  dof={dof}')
print(f'  Result : {"SIGNIFICANT" if p < 0.05 else "NOT significant"} (a=0.05)')

rates = df.groupby('hypertension')['stroke'].mean() * 100
fig, ax = plt.subplots(figsize=(6, 4))
rates.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'])
ax.set_title('RQ2 -- Stroke Rate by Hypertension (%)')
ax.set_xticklabels(['No Hypertension', 'Hypertension'], rotation=0)
ax.set_ylabel('Stroke Rate %')
plt.tight_layout()
plt.show()

## RQ3: Do Some Departments (Work Types) Experience Higher Turnover (Stroke)? — ANOVA

In [ ]:
groups = [df[df['work_type'] == wt]['stroke'].values for wt in df['work_type'].unique()]
f_stat, p_val = stats.f_oneway(*groups)

print('RQ3 -- ANOVA: Work Type vs Stroke Rate')
rates = df.groupby('work_type')['stroke'].mean().mul(100).round(2).sort_values(ascending=False)
print(rates.to_string())
print(f'\n  F={f_stat:.4f}  p={p_val:.4e}')
print(f'  Result : {"SIGNIFICANT" if p_val < 0.05 else "NOT significant"} (a=0.05)')

fig, ax = plt.subplots()
rates.plot(kind='bar', ax=ax, color='mediumpurple')
ax.set_title('RQ3 -- Stroke Rate by Work Type / Department (ANOVA)')
ax.set_ylabel('Stroke Rate %')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## RQ4: Does Workload (BMI) Affect Resignation (Stroke) Rates? — Chi-Square + Quartiles

In [ ]:
df_bmi = df.dropna(subset=['bmi']).copy()
df_bmi['bmi_quartile'] = pd.qcut(df_bmi['bmi'], q=4, labels=['Q1 Low','Q2','Q3','Q4 High'])

ct = pd.crosstab(df_bmi['bmi_quartile'], df_bmi['stroke'])
chi2, p, dof, _ = stats.chi2_contingency(ct)

print('RQ4 -- Chi-Square + Quartiles: BMI vs Stroke')
print(ct)
print(f'\n  Chi2={chi2:.4f}  p={p:.4e}  dof={dof}')
print(f'  Result : {"SIGNIFICANT" if p < 0.05 else "NOT significant"} (a=0.05)')
print('\nStroke rate per BMI quartile:')
print(df_bmi.groupby('bmi_quartile', observed=True)['stroke'].mean().mul(100).round(2))

fig, ax = plt.subplots()
df_bmi.groupby('bmi_quartile', observed=True)['stroke'].mean().mul(100).plot(kind='bar', ax=ax, color='darkorange')
ax.set_title('RQ4 -- Stroke Rate by BMI Quartile (Chi-Square + Quartiles)')
ax.set_ylabel('Stroke Rate %')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## RQ5: Which Employee Groups Are Highest Risk? — Employee Segmentation

In [ ]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 30, 60, 120],
    labels=['Young (<30)', 'Middle-aged (30-60)', 'Senior (>60)']
)

df['risk_score'] = (
    df['hypertension'] +
    df['heart_disease'] +
    (df['avg_glucose_level'] > 140).astype(int) +
    (df['age'] > 60).astype(int) +
    df['smoking_status'].isin(['formerly smoked', 'smokes']).astype(int)
)

seg = df.groupby('age_group', observed=True).agg(
    total=('patient_id', 'count'),
    stroke_cases=('stroke', 'sum'),
    stroke_rate_pct=('stroke', lambda x: round(x.mean() * 100, 2)),
    avg_risk_score=('risk_score', 'mean')
).sort_values('stroke_rate_pct', ascending=False)

print('RQ5 — Employee Segmentation by Age Group')
print(seg.to_string())

high_risk = df[df['risk_score'] >= 3]
print(f'\nHigh-risk patients (score ≥ 3): {len(high_risk)}')
print(f'Stroke rate in high-risk group : {high_risk["stroke"].mean()*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
seg['stroke_rate_pct'].plot(kind='bar', ax=axes[0], color=['#e74c3c','#f39c12','#2ecc71'])
axes[0].set_title('RQ5 — Stroke Rate by Age Group (Segmentation)')
axes[0].set_ylabel('Stroke Rate %')
axes[0].tick_params(axis='x', rotation=15)

df.groupby('risk_score')['stroke'].mean().mul(100).plot(kind='bar', ax=axes[1], color='crimson')
axes[1].set_title('Stroke Rate by Risk Score (0–5)')
axes[1].set_ylabel('Stroke Rate %')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## Retention Dashboard — Executive Summary

In [ ]:
print('=' * 55)
print('        RETENTION DASHBOARD — EXECUTIVE SUMMARY')
print('=' * 55)
print(f'  Total patients       : {len(df):,}')
print(f'  Overall stroke rate  : {df["stroke"].mean()*100:.2f}%')
print()
print('  Department Analysis (RQ3 — ANOVA):')
for wt, rate in df.groupby('work_type')['stroke'].mean().mul(100).sort_values(ascending=False).items():
    print(f'    {wt:20s}: {rate:.2f}%')
print()
print('  Executive Recommendations:')
print('    - Senior patients (>60) have 13.5% stroke rate — prioritize screening')
print('    - Hypertension doubles stroke risk — target early intervention')
print('    - High glucose (>140) is a significant independent risk factor')
print('    - High-risk patients (score≥3) need immediate follow-up')
print('=' * 55)

conn.close()